# 01 – Market Discovery & Triad Selection

This notebook displays the **9 resolved Polymarket markets** automatically
selected across 3 triads (politics, crypto, sports).  Each triad contains
a **base**, **similar**, and **dissimilar** market.

Selection is performed by `src/collect/select_markets.py` which queries
the Gamma search API for high-volume resolved markets in diverse domains.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_colwidth", 90)

from src.config import TRIADS, ALL_MARKETS

## Selected Markets Overview

In [2]:
mkt_df = pd.DataFrame(ALL_MARKETS)
mkt_df[["triad_id", "role", "category", "slug", "end_date"]]

,triad_id,role,category,slug,end_date
0,politics,base,politics,will-donald-trump-win-the-2024-us-presidential-election,2024-11-05T12:00:00Z
1,politics,similar,politics,will-kamala-harris-win-the-popular-vote-in-the-2024-presidential-election,2024-11-05T12:00:00Z
2,politics,dissimilar,crypto,will-bitcoin-hit-100k-in-2024,2024-12-30T12:00:00Z
3,crypto,base,crypto,will-bitcoin-hit-100k-again-in-2024-dec-23,2024-12-31T12:00:00Z
4,crypto,similar,crypto,will-microstrategy-purchase-more-bitcoin-in-2024-dec-16,2024-12-31T00:00:00Z
5,crypto,dissimilar,entertainment,oscars-best-picture-will-anora-win-best-picture-at-the-2025-oscars,2025-03-02T12:00:00Z
6,sports,base,sports,will-saquon-barkley-win-super-bowl-lix-mvp,2025-02-09T12:00:00Z
7,sports,similar,sports,will-shai-gilgeous-alexander-win-the-2025-nba-finals-mvp,2025-06-24T12:00:00Z
8,sports,dissimilar,politics,will-nikki-haley-win-the-2024-republican-iowa-caucus,2024-01-15T00:00:00Z


## Per-Triad Detail

In [3]:
for tid, markets in TRIADS.items():
    print(f"\n{'═' * 70}")
    print(f"  Triad: {tid}")
    print(f"{'═' * 70}")
    for m in markets:
        print(f"  [{m['role']:11s}]  {m['category']:14s}  end={m['end_date'][:10]}")
        print(f"               slug = {m['slug']}")


══════════════════════════════════════════════════════════════════════
  Triad: politics
══════════════════════════════════════════════════════════════════════
  [base       ]  politics        end=2024-11-05
               slug = will-donald-trump-win-the-2024-us-presidential-election
  [similar    ]  politics        end=2024-11-05
               slug = will-kamala-harris-win-the-popular-vote-in-the-2024-presidential-election
  [dissimilar ]  crypto          end=2024-12-30
               slug = will-bitcoin-hit-100k-in-2024

══════════════════════════════════════════════════════════════════════
  Triad: crypto
══════════════════════════════════════════════════════════════════════
  [base       ]  crypto          end=2024-12-31
               slug = will-bitcoin-hit-100k-again-in-2024-dec-23
  [similar    ]  crypto          end=2024-12-31
               slug = will-microstrategy-purchase-more-bitcoin-in-2024-dec-16
  [dissimilar ]  entertainment   end=2025-03-02
               slug = o

## Data Availability Check

Verify that price and trade data has been collected for all 9 markets.

In [6]:
from src.config import DATA_PROCESSED

rows = []
for m in ALL_MARKETS:
    slug = m["slug"]
    hourly = DATA_PROCESSED / f"prices_{slug}_hourly.parquet"
    hf     = DATA_PROCESSED / f"prices_{slug}_highfreq.parquet"
    trades = DATA_PROCESSED / f"trades_{slug}.parquet"
    
    def _count(p):
        if p.exists():
            return len(pd.read_parquet(p))
        return 0
    
    rows.append({
        "triad": m["triad_id"],
        "role": m["role"],
        "slug": slug[:50],
        "hourly_rows": _count(hourly),
        "highfreq_rows": _count(hf),
        "trade_rows": _count(trades),
    })

avail = pd.DataFrame(rows)
avail.style.background_gradient(cmap="YlGn", subset=["hourly_rows", "highfreq_rows", "trade_rows"])

,triad,role,slug,hourly_rows,highfreq_rows,trade_rows
0,politics,base,will-donald-trump-win-the-2024-us-presidential-ele,7333,4030,1000
1,politics,similar,will-kamala-harris-win-the-popular-vote-in-the-202,7212,4030,1000
2,politics,dissimilar,will-bitcoin-hit-100k-in-2024,6613,0,1000
3,crypto,base,will-bitcoin-hit-100k-again-in-2024-dec-23,184,2208,1000
4,crypto,similar,will-microstrategy-purchase-more-bitcoin-in-2024-d,167,1911,1000
5,crypto,dissimilar,oscars-best-picture-will-anora-win-best-picture-at,951,4032,1000
6,sports,base,will-saquon-barkley-win-super-bowl-lix-mvp,376,4032,483
7,sports,similar,will-shai-gilgeous-alexander-win-the-2025-nba-fina,1497,3696,1000
8,sports,dissimilar,will-nikki-haley-win-the-2024-republican-iowa-cauc,145,1734,91
